# 실습 02 · 고전 NMS와 매칭 분리

> **Day15 | 객체탐지·YOLO·추적 · 02_응용**
> 원본 스크립트: `02_nms.py` · 교안 PPT 27쪽 · 직접 구현: A02
> 결과: 이 노트북을 실제 실행한 출력

## 학습 목표

- `batched_nms`로 같은 클래스끼리만 중복 후보를 제거한다.
- NMS(후보 정리)와 매칭(정답 비교)이 다른 단계임을 구분한다.
- conf와 NMS IoU를 하나씩 바꿔 결과 차이를 비교한다.

## 핵심 개념

| 개념 | 설명 |
|---|---|
| NMS | 겹치는 낮은 점수 후보 제거 |
| batched_nms | idxs(클래스)가 같은 후보끼리만 억제 |
| YOLO26 | 기본 추론이 end-to-end(NMS-free) → 이 실습은 원리 입력으로 수행 |

## 0. 준비

- 이 노트북은 `02_응용` 폴더(같은 폴더에 `data/`가 있는 위치)에서 연다.
- 입력은 `data/`의 원본을 읽기만 하고, 노트북이 만드는 결과 파일은 `outputs_notebook/`에 저장한다.
- CPU에서 실행한다. GPU·CUDA 코드는 없다.

In [1]:
# ============================================================
# 0. 라이브러리·경로 설정
# ============================================================
from pathlib import Path
import logging
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BASE : 노트북 폴더(02_응용) / DATA : 원본 입력 / OUT : 노트북 결과 폴더
BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "outputs_notebook"
OUT.mkdir(exist_ok=True)
# ultralytics 로그는 WARNING 이상만 출력
logging.getLogger("ultralytics").setLevel(logging.WARNING)

def show(images, titles, size=3.2):
    """BGR 배열 여러 장을 한 줄로 표시한다.

    - images: BGR 배열 목록
    - titles: 각 그림 제목 목록 (영문)
    """
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.9))
    axes = np.atleast_1d(axes)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

## 1. 후보·정답 Tensor

In [2]:
# ============================================================
# 1. 후보·정답 Tensor
# ============================================================
import torch
from torchvision.ops import box_iou, batched_nms

p = pd.read_csv(DATA/"boxes_fixture.csv"); g = pd.read_csv(DATA/"boxes_gt.csv")
boxes = torch.tensor(p[["x1", "y1", "x2", "y2"]].to_numpy(), dtype=torch.float32)
scores = torch.tensor(p.confidence.to_numpy(), dtype=torch.float32)
classes = torch.tensor(p.class_id.to_numpy(), dtype=torch.int64)
gb = torch.tensor(g[["x1", "y1", "x2", "y2"]].to_numpy(), dtype=torch.float32)
print("후보끼리 IoU 표:")
print(box_iou(boxes, boxes).numpy().round(3))

후보끼리 IoU 표:
[[1.  0.9 0.  0. ]
 [0.9 1.  0.  0. ]
 [0.  0.  1.  0. ]
 [0.  0.  0.  1. ]]


## 2. 조건별 NMS → 매칭 (TODO A02)

`batched_nms`의 반환 index는 `ids`로 줄인 배열 기준이므로 `ids[...]`로 원본 index로 바꾼다.

In [3]:
# ============================================================
# 2. 조건별 NMS -> 매칭 (TODO A02)
# ============================================================
rows = []
for conf, nms_iou in [(0.25, 0.5), (0.5, 0.5), (0.25, 0.95)]:
    ids = torch.where(scores >= conf)[0]
    # A02 : 같은 클래스끼리만 억제 -> 원본 index로 변환
    keep = ids[batched_nms(boxes[ids], scores[ids], classes[ids], nms_iou)]
    used = set(); tp = 0
    for i in keep.tolist():
        ovs = box_iou(boxes[[i]], gb)[0]
        eligible = [j for j in range(len(g)) if int(classes[i]) == int(g.iloc[j].class_id) and j not in used]
        for j in sorted(eligible, key=lambda j: -float(ovs[j])):
            if ovs[j] >= 0.5: used.add(j); tp += 1; break
    n = len(keep)
    rows.append(dict(conf=conf, nms_iou=nms_iou, kept=n, kept_index=keep.tolist(), tp=tp, fp=n-tp, fn=len(g)-tp, precision=tp/n if n else 0, recall=tp/len(g)))
r = pd.DataFrame(rows); r.drop(columns="kept_index").to_csv(OUT/"nms_comparison.csv", index=False)
r.round(4)

,conf,nms_iou,kept,kept_index,tp,fp,fn,precision,recall
0,0.25,0.50,3,"[0, 2, 3]",2,1,1,0.6667,0.6667
1,0.50,0.50,2,"[0, 2]",2,0,1,1.0000,0.6667
2,0.25,0.95,4,"[0, 1, 2, 3]",2,2,1,0.5000,0.6667


## 3. 클래스 구분 없이 NMS를 하면

`torchvision.ops.nms`는 클래스를 보지 않는다. 이 입력에서는 다른 클래스끼리 겹치는 후보가 없어 결과가 같지만, 다른 클래스가 겹치면 정상 후보까지 지워질 수 있다.

In [4]:
# ============================================================
# 3. 클래스 무시 NMS와 비교
# ============================================================
from torchvision.ops import nms
print("batched_nms (IoU 0.5):", batched_nms(boxes, scores, classes, 0.5).tolist())
print("nms 클래스 무시 (IoU 0.5):", nms(boxes, scores, 0.5).tolist())
# 다른 클래스 후보가 같은 위치에 있는 가상 예
b2 = torch.tensor([[0, 0, 10, 10], [0, 0, 10, 10]], dtype=torch.float32); s2 = torch.tensor([0.9, 0.8]); c2 = torch.tensor([0, 1])
print("겹친 두 클래스 -> batched_nms:", batched_nms(b2, s2, c2, 0.5).tolist(), "| nms:", nms(b2, s2, 0.5).tolist())

batched_nms (IoU 0.5): [0, 2, 3]
nms 클래스 무시 (IoU 0.5): [0, 2, 3]
겹친 두 클래스 -> batched_nms: [0, 1] | nms: [0]


## 정리 (README 설명 질문)

- **무엇을 고정했고 무엇을 바꾸었는가?** 후보 4개·정답 3개·매칭 IoU 0.5를 고정했다. 1행 대비 2행은 conf만(0.25→0.5), 3행은 NMS IoU만(0.5→0.95) 바꿨다. conf 0.5는 0.40 triangle 오검출을 없애 precision 1.0, NMS IoU 0.95는 중복 square(후보끼리 IoU 0.90)를 지우지 못해 FP 2.
- **결과만으로 단정할 수 없는 부분은?** 손으로 만든 후보 4개라 실제 모델의 conf·NMS 최적값을 말할 수 없다. 또 YOLO26 기본 추론은 NMS-free라 `iou` 인자를 바꿔도 같은 효과가 난다고 단정할 수 없다.